<a href="https://colab.research.google.com/github/JONATHAN-2009/APP/blob/main/notebooks/ACE_Step_1_5_Music_Generator_CustomUI_by_AIQUEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>&#127925; ACE-Step 1.5 AI Music Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Custom UI Edition - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0 0 20px 0;'>Free on Google Colab T4 GPU | MIT License</p>

  ---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-Free%20Tier-orange?style=for-the-badge&logo=googlecolab&logoColor=white" />

 <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/▶%20Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20𝕏-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---


### What is ACE-Step 1.5?

**ACE-Step 1.5** is a powerful open-source music generation model that creates complete songs from text prompts and lyrics.

| Feature | Details |
|---------|----------|
| Speed | Generate music in seconds |
| Languages | 50+ languages supported |
| Modes | Text2Music, Cover, Repaint |
| License | MIT - Free for commercial use |
| Control | BPM, key, duration, time signature |

### Before You Start
> Go to **Runtime -> Change runtime type -> T4 GPU**

---
## Step 1: Verify GPU & Install PyTorch

Checks your GPU and installs PyTorch 2.3.1 to avoid CUDA OOM errors.

> If no GPU is detected -> **Runtime -> Change runtime type -> T4 GPU**

In [1]:
# @title Step 1: Verify GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print()

import torch
print(f"  PyTorch {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07

  PyTorch 2.10.0+cu128
  CUDA available: True
  GPU: Tesla T4
  VRAM: 14.6 GB


In [2]:
# @title Step 2: Install uv
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 11), "ACE-Step 1.5 requires Python >= 3.11"

print("\nInstalling uv package manager...")
!curl -LsSf https://astral.sh/uv/install.sh | sh 2>&1 | tail -1

import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version
print("Done!")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

Installing uv package manager...
everything's installed!
uv 0.10.8
Done!


In [3]:
# @title Step 3: Clone & Install
%cd /content

import os
if os.path.exists("/content/ACE-Step-1.5"):
    print("Repository exists, pulling latest...")
    %cd /content/ACE-Step-1.5
    !git pull
else:
    print("Cloning ACE-Step 1.5...")
    !git clone https://github.com/ACE-Step/ACE-Step-1.5.git
    %cd /content/ACE-Step-1.5

print("\nInstalling dependencies (this may take 2-3 minutes)...\n")
!uv sync
print("\nAll dependencies installed!")

/content
Cloning ACE-Step 1.5...
Cloning into 'ACE-Step-1.5'...
remote: Enumerating objects: 7997, done.
remote: Counting objects: 100% (2728/2728), done.
remote: Compressing objects: 100% (715/715), done.
remote: Total 7997 (delta 2268), reused 2020 (delta 2013), pack-reused 5269 (from 3)
Receiving objects: 100% (7997/7997), 10.90 MiB | 24.58 MiB/s, done.
Resolving deltas: 100% (4821/4821), done.
/content/ACE-Step-1.5

Installing dependencies (this may take 2-3 minutes)...

Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 173 packages in 18.44s
Prepared 139 packages in 1m 42s
Installed 139 packages in 1.41s
 + absl-py==2.4.0
 + accelerate==1.13.0
 + ace-step==1.5.0 (from file:///content/ACE-Step-1.5)
 + aiofiles==24.1.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.12.1
 + attrs==25.4.0
 + brotli==1.2.0
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-

In [ ]:
# @title Step 4: Launch Music Generator
%cd /content/ACE-Step-1.5

import os
os.environ["TORCH_CUDNN_SDPA_ENABLED"] = "1"
os.environ["ATTN_BACKEND"] = "sdpa"
os.environ["MPLBACKEND"] = "agg"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

LAUNCHER = '\nimport sys, os, importlib, pkgutil, traceback\nos.environ["MPLBACKEND"] = "agg"\n\ndef run_custom_ui(dit, llm, **kwargs):\n    print("🎨 Launching Custom Music Studio UI...")\n    import gradio as gr\n    import random\n    from acestep.inference import generate_music, GenerationParams, GenerationConfig, format_sample\n\n    try:\n        from acestep.gradio_ui.ui_components import EXAMPLE_POOL\n        print(f"✅ Loaded {len(EXAMPLE_POOL)} examples from original UI")\n    except:\n        try:\n            from acestep.gradio_ui.examples import get_examples\n            EXAMPLE_POOL = get_examples()\n            print(f"✅ Loaded {len(EXAMPLE_POOL)} examples")\n        except:\n            EXAMPLE_POOL = [\n                {"caption": "aggressive, high-energy heavy metal, dual heavily distorted guitars, chugging riffs, powerful driving drums, male vocals forceful raspy, melodic technical guitar solo, fast runs expressive bends, clean powerful production, anthemic rebellious",\n                 "lyrics": "[Intro - Guitar Riff]\\n\\n[Verse 1]\\nRip the script, break it mold\\nShatter chains, dare the bold\\nTook your crown, you can\'t ignore\\nFreedom\'s knock on every door\\n\\n[Chorus]\\nBreak the chains, let it pour\\nShout it loud, beg no more\\nBurn the rules, end the war\\nWe\'re alive, can\'t ignore\\nCan\'t ignore\\n\\n[Verse 2]\\nTear it down, brick by brick\\nEvery lie, every trick\\nSkies on fire, won\'t obey\\nChaos calls, don\'t delay\\n\\n[Chorus]\\nBreak the chains, let it pour\\nShout it loud, beg no more\\nBurn the rules, end the war\\nWe\'re alive, can\'t ignore\\nCan\'t ignore\\n\\n[Guitar Solo]\\n\\n[Verse 3]\\nEchoes scream, hearts collide\\nNo surrender, no divide\\nWolves are howling, hear their cry\\nWe rise up, won\'t comply\\n\\n[Chorus]\\nBreak the chains, let it pour\\nShout it loud, beg no more\\nBurn the rules, end the war\\nWe\'re alive, can\'t ignore\\nCan\'t ignore\\n\\n[Outro - Guitar Riff]\\n[abrupt silence]",\n                 "language": "en"},\n                {"caption": "female vocals, rap, modern, hip hop, Indian fusion, whispered, plucked synth melody, drums of earth",\n                 "lyrics": "[Intro - Plucked Synth Melody]\\n\\n[Verse 1]\\nThe sun melts over the endless plain\\nA mirage dances like a golden chain\\nBare feet trace stories in the cracked dry land\\nWho holds the map in this empty game?\\n\\n[Chorus]\\nDesert echoes\\nThey call my name\\nDrums of the earth, they beat the same\\nStep to the rhythm, no one to blame\\nDesert echoes\\nDesert echoes\\n\\n[Instrumental Break - Plucked Synth Melody]\\n\\n[Verse 2]\\nA lone wind whispers secrets untold\\nStars blink brighter than the city\'s gold\\nMy shadow stretches long in the moonlit fold\\nThis is my world, fierce and bold\\n\\n[Bridge]\\nDo you hear it? The silence screams\\nDid you see it? The shifting dreams\\n[Vocal chop: \'dreams\']\\n\\n[Chorus]\\nDesert echoes\\nThey call my name\\nDrums of the earth, they beat the same\\nStep to the rhythm, no one to blame\\nDesert echoes\\n\\n[Plucked synth melody fades out]",\n                 "language": "en"},\n                {"caption": "romantic active synthpop anthem, joyful energy, irresistible charm, bright engaging female vocals, feelgood danceable",\n                 "lyrics": "Midnight talks with my regrets,\\nWhiskey lips and cigarettes.\\nPromises like shattered glass,\\nCut me deep but never last.\\n\\nI danced with fire, played the fool,\\nBroke the rules and called it cool.\\nNow the mirror wont lie\\nI dont recognize these eyes.\\n\\nI was right, I was left\\nCaught between the lines, no regrets\\nI was wrong, I was left behind\\nLike a shadow fading with the light\\n\\nI\'ve been up, but mostly down\\nChasing dreams that hit the ground\\nEvery high just slips away\\nLeaving echoes of my name\\n\\nI was right, I was left,\\nTorn in two with every step.\\nI was wrong, I was left behind,\\nAnother love I couldnt find.\\n\\nIve been up, but mostly down,\\nChasing highs that let me drown.\\nI was right, I was left,\\nNow Im at a new low.",\n                 "language": "en"},\n                {"caption": "smooth jazzy lo-fi hip-hop, gentle piano melody, relaxed steady drum machine, warm round bassline, duet clear melodic female and smooth conversational male vocals, tasteful melodic saxophone fills, late-night atmosphere, extended instrumental outro, expressive improvisational sax solo",\n                 "lyrics": "[Intro: Piano & Saxophone]\\n\\n[Verse 1: Female Vocal]\\n\\u6708\\u5149\\u843d\\u8fdb\\u7a97\\n\\u50cf\\u662f\\u8d77\\u8272\\u7684\\u90a3\\u676f\\n\\u4f60\\u8fd8\\u5728\\u6211\\u7684\\u5fc3\\u5e95\\u65cb\\u8f6c\\n\\u629b\\u5f00\\u6240\\u6709\\u70e6\\u5fe7\\n\\u8fd9\\u7231\\u50cf\\u6d41\\u661f\\u5760\\u843d\\n\\u5728\\u591c\\u7a7a\\u95ea\\u8000\\n\\u65e0\\u9700\\u89e3\\u7b54 \\u5fc3\\u52a8\\u662f\\u552f\\u4e00\\u7684\\u89e3\\u836f\\n\\n[Verse 2: Male Vocal]\\n\\u4f60\\u7684\\u773c\\u795e\\u50cf\\u8ff7\\u5bab\\n\\u6211\\u5f98\\u5f8a\\u5176\\u4e2d\\n\\u6bcf\\u4e2a\\u97f3\\u7b26\\u90fd\\u5e26\\u4e86\\u5fc3\\u810f\\u7684\\u8f70\\u9e23\\n\\u89e6\\u78b0\\u4f60\\u7684\\u6e29\\u5ea6\\u5c31\\u50cf\\u7535\\u6d41\\u7a7f\\u5fc3\\n\\u65e0\\u6cd5\\u6297\\u62d2\\u4f60\\u7684\\u7231\\u628a\\u6211\\u62c9\\u5165\\u68a6\\u5883\\n\\n[Chorus: Duet]\\n\\u5fc3\\u8df3\\u968f\\u7740\\u4f60\\u72c2\\u5954\\n\\u5982\\u6d77\\u6f6e\\u6c79\\u6d8c\\n\\u7231\\u7684\\u8282\\u594f\\u7ffb\\u6eda\\u7740\\n\\u50cf\\u68a6\\u5883\\u4e2d\\u68a6\\n\\u4f60\\u7684\\u540d\\u5b57\\u662f\\u65cb\\u5f8b\\n\\u65cb\\u5f8b\\u4e2d\\u6210\\u98ce\\n\\u6bcf\\u4e00\\u62cd\\u90fd\\u8bc9\\u8bf4\\u7231\\u65e0\\u58f0\\u7684\\u60b8\\u52a8\\n\\n[Instrumental Break: Saxophone Solo]\\n\\n[Verse 3: Male Vocal]\\n\\u9ed1\\u591c\\u50cf\\u4e2a\\u821e\\u53f0\\n\\u6211\\u4eec\\u70b9\\u4e86\\u5f69\\u706f\\n\\u8282\\u62cd\\u8ddf\\u968f\\u5fc3\\u8df3\\n\\u9ed8\\u5951\\u878d\\u6210\\u4e00\\u543b\\n\\u522b\\u8ba9\\u591c\\u665a\\u505c\\u4e0b\\u6765\\n\\u7231\\u6c38\\u8fdc\\u5728\\u8ba4\\u771f\\n\\u6bcf\\u4e00\\u79d2\\u90fd\\u50cf\\u6b4c\\n\\u5f00\\u542f\\u5947\\u5999\\u4eba\\u751f\\n\\n[Chorus: Duet]\\n\\u5fc3\\u8df3\\u968f\\u7740\\u4f60\\u72c2\\u5954\\n\\u5982\\u6d77\\u6f6e\\u6c39\\u6d8c\\n\\u7231\\u7684\\u8282\\u594f\\u7ffb\\u6eda\\u7740\\n\\u50cf\\u68a6\\u5883\\u4e2d\\u68a6\\n\\u4f60\\u7684\\u540d\\u5b57\\u662f\\u65cb\\u5f8b\\n\\u65cb\\u5f8b\\u4e2d\\u6210\\u98ce\\n\\u6bcf\\u4e00\\u62cd\\u90fd\\u8bc9\\u8bf4\\u7231\\u65e0\\u58f0\\u7684\\u60b8\\u52a8\\n\\n[Outro: Duet & Saxophone]\\n[Saxophone melody fades out]",\n                 "language": "zh"},\n                {"caption": "explosive high-energy K-pop EDM, relentless four-on-the-floor beat, pulsing synth bassline, bright piano melody, shimmering arpeggiated synths, powerful clear male lead vocal, anthemic melody, energetic ad-libs hype-man shouts",\n                 "lyrics": "[Intro]\\n[Synth arpeggio and pads]\\n\\n[Verse 1]\\n[ko] hwangholhan i jilseo-e neon ppajyeo\\n[en] (oh yeah)\\n[ko] lideum wi-e sinhoneul jilleo\\n[en] (oh, let\'s go)\\n\\n[Chorus]\\n[ko] oelo-un i jilseo gip-eojyeo\\n[en] (yeah, yeah)\\n[ko] lideum wi-e him-eul jwi-yeo\\n[en] (let\'s go)\\n\\n[Instrumental Break]\\n[Synth lead melody with vocal chops]\\n\\n[Outro]\\n[abrupt silence]",\n                 "language": "ko"},\n                {"caption": "rap, minimal, spoken word, dark, minimal beats heavy basslines, sharp snares, female vocals, introspective raw energy",\n                 "lyrics": "[Intro]\\n[Music box synth]\\n[whispered] Yeah\\n\\n[Verse 1]\\n\\u9019\\u4e0d\\u662f\\u6211\\u7684\\u8857 \\u6211\\u7684\\u6839\\n\\u5730\\u4e0b\\u7684 pum pum \\u50cf\\u5fc3\\u8df3\\n\\u6572\\u97ff\\u90a3\\u5f8b\\u52d5 \\u6211\\u7684\\u5144\\u5f1f\\n\\n[Chorus]\\n\\u4e0d\\u662f\\u62b1\\u6028 \\u53ea\\u662f\\u770b\\u6e05\\n\\u5728\\u9019\\u8ff7\\u5e7b\\u7684\\u591c\\u88e1\\u601d\\u8003\\u4eba\\u751f\\u7684\\u610f\\u7fa9\\n\\n[Outro]\\n[Beat fades]",\n                 "language": "zh"},\n                {"caption": "driving post-punk, layered electric guitars, male lead vocal angsty strained, anthemic shouted chorus",\n                 "lyrics": "[Intro - Guitar]\\n\\n[Verse 1]\\nUnder neon lights, they flicker and fade\\nLost in the hum of this restless parade\\n\\n[Chorus]\\nOh, shadows of neon\\nWhere do you run?\\n\\n[Guitar Solo]\\n\\n[Outro - fade]",\n                 "language": "en"},\n                {"caption": "classic country-folk ballad, steady acoustic guitar, warm male vocal gentle twang, plaintive harmonica",\n                 "lyrics": "[Intro - Acoustic Guitar]\\n\\n[Verse 1]\\nFound you standing in the river\'s roar\\nHeartbeats racing like never before\\n\\n[Chorus]\\nI\'ll ride these heartstrings all night long\\nSing our song where we belong\\n\\n[Outro - Harmonica]",\n                 "language": "en"},\n                {"caption": "Funk, Soul, Groove, Male Vocals, Slap Bass, Horn Section, 1970s, Party Energy",\n                 "lyrics": "[Drum Break]\\nGet on up!\\n\\n[Verse 1]\\nI feel the rhythm moving in my feet\\n\\n[Chorus]\\nGive it up, turn it loose!\\nWe got the funk!",\n                 "language": "en"},\n                {"caption": "symphonic metal epic duet, female soprano male baritone, orchestral strings, operatic choirs",\n                 "lyrics": "[Intro - Orchestral]\\n\\n[Verse 1 - Female]\\nI see you walking through the ash\\n\\n[Chorus - Duet]\\nWe stand upon the edge of the night\\n\\n[Outro - fade to black]",\n                 "language": "en"},\n            ]\n            print(f"✅ Loaded {len(EXAMPLE_POOL)} custom examples")\n\n    LANG = {\n        "en": "🇬🇧 English", "zh": "🇨🇳 Chinese", "ja": "🇯🇵 Japanese", "ko": "🇰🇷 Korean", "es": "🇪🇸 Spanish",\n        "fr": "🇫🇷 French", "de": "🇩🇪 German", "it": "🇮🇹 Italian", "pt": "🇵🇹 Portuguese", "ru": "🇷🇺 Russian",\n        "ar": "🇸🇦 Arabic", "hi": "🇮🇳 Hindi", "tr": "🇹🇷 Turkish", "pl": "🇵🇱 Polish", "nl": "🇳🇱 Dutch",\n        "sv": "🇸🇪 Swedish", "no": "🇳🇴 Norwegian", "da": "🇩🇰 Danish", "fi": "🇫🇮 Finnish", "el": "🇬🇷 Greek",\n        "he": "🇮🇱 Hebrew", "th": "🇹🇭 Thai", "vi": "🇻🇳 Vietnamese", "id": "🇮🇩 Indonesian", "ms": "🇲🇾 Malay",\n        "tl": "🇵🇭 Tagalog", "uk": "🇺🇦 Ukrainian", "cs": "🇨🇿 Czech", "hu": "🇭🇺 Hungarian", "ro": "🇷🇴 Romanian",\n        "bg": "🇧🇬 Bulgarian", "hr": "🇭🇷 Croatian", "sr": "🇷🇸 Serbian", "sk": "🇸🇰 Slovak", "sl": "🇸🇮 Slovenian",\n        "lt": "🇱🇹 Lithuanian", "lv": "🇱🇻 Latvian", "et": "🇪🇪 Estonian", "fa": "🇮🇷 Persian", "ur": "🇵🇰 Urdu",\n        "bn": "🇧🇩 Bengali", "ta": "🇮🇳 Tamil", "te": "🇮🇳 Telugu", "mr": "🇮🇳 Marathi", "gu": "🇮🇳 Gujarati",\n        "kn": "🇮🇳 Kannada", "ml": "🇮🇳 Malayalam", "pa": "🇮🇳 Punjabi", "sw": "🇰🇪 Swahili", "af": "🇿🇦 Afrikaans"\n    }\n\n    def lc(l): return {v: k for k, v in LANG.items()}.get(l, "en")\n\n    def detect_language(lyrics):\n        if not lyrics or lyrics.strip() == "[inst]": return "en"\n        text = lyrics.lower()\n        if any(char in text for char in [\'\\u661f\', \'\\u6708\', \'\\u7231\', \'\\u68a6\', \'\\u5fc3\']): return "zh"\n        if any(char in text for char in [\'\\u306e\', \'\\u3092\', \'\\u306b\', \'\\u306f\', \'\\u304c\']): return "ja"\n        if any(char in text for char in [\'\\uc774\', \'\\uadf8\', \'\\uc800\', \'\\ub97c\', \'\\uc740\']): return "ko"\n        if any(word in text for word in [\'esta\', \'amor\', \'noche\']): return "es"\n        if any(char in text for char in [\'\\u062a\', \'\\u0646\', \'\\u0645\', \'\\u0644\', \'\\u0639\']): return "ar"\n        if any(char in text for char in [\'\\u0e09\', \'\\u0e40\', \'\\u0e32\', \'\\u0e21\', \'\\u0e01\']): return "th"\n        if any(word in text for word in [\'eg\', \'og\', \'g\\u00e5r\']): return "no"\n        return "en"\n\n    PROJECT_ROOT = "/content/ACE-Step-1.5"\n    OUTPUT_DIR = os.path.join(PROJECT_ROOT, "gradio_outputs")\n    os.makedirs(OUTPUT_DIR, exist_ok=True)\n\n    def wrap_init_engine(p=gr.Progress()):\n        p(0.1, desc="Checking GPU...")\n        print("🚀 Initializing Engine...")\n        init_kwargs = {\n            "project_root": PROJECT_ROOT,\n            "config_path": "acestep-v15-turbo",\n            "device": "cuda",\n            "offload_to_cpu": True,\n            "compile_model": False,\n            "use_flash_attention": False\n        }\n        try:\n            p(0.3, desc="Loading DiT...")\n            status, ok = dit.initialize_service(**init_kwargs)\n            if not ok:\n                raise RuntimeError(f"DiT Init Failed: {status}")\n            print("✅ DiT loaded")\n        except Exception as e:\n            raise gr.Error(f"Failed to initialize DiT: {e}")\n        p(0.5, desc="Checking LLM model...")\n        lm_model_name = "acestep-5Hz-lm-0.6B"\n        lm_checkpoint_dir = os.path.join(PROJECT_ROOT, "checkpoints")\n        lm_model_dir = os.path.join(lm_checkpoint_dir, lm_model_name)\n        lm_loaded = False\n\n        # Auto-download 0.6B model from HuggingFace if not present\n        if not os.path.exists(lm_model_dir):\n            print(f"📥 Downloading {lm_model_name} from HuggingFace...")\n            p(0.55, desc=f"Downloading {lm_model_name}...")\n            try:\n                from huggingface_hub import snapshot_download\n                snapshot_download(\n                    repo_id=f"ACE-Step/{lm_model_name}",\n                    local_dir=lm_model_dir,\n                    local_dir_use_symlinks=False,\n                )\n                print(f"✅ Downloaded {lm_model_name}")\n            except Exception as dl_e:\n                print(f"⚠️ Failed to download LLM: {dl_e}")\n\n        p(0.7, desc="Loading LLM (0.6B via PyTorch)...")\n        try:\n            lm_status, lm_ok = llm.initialize(\n                checkpoint_dir=lm_checkpoint_dir,\n                lm_model_path=lm_model_name,\n                backend="pt",\n                device="cuda",\n                offload_to_cpu=True\n            )\n            if lm_ok:\n                lm_loaded = True\n                print("✅ LLM loaded successfully!")\n            else:\n                print(f"⚠️ LLM init returned: {lm_status}")\n        except Exception as e:\n            print(f"⚠️ LLM failed: {e}")\n            traceback.print_exc()\n\n        p(1.0, desc="Ready!")\n        suffix = " + LLM (0.6B CoT)" if lm_loaded else " (DiT only, no LLM)"\n        return f"✅ Engine Ready!{suffix}", gr.update(interactive=True), gr.update(visible=False)\n\n    def randomize_example():\n        example = random.choice(EXAMPLE_POOL)\n        caption = example.get("caption", "")\n        lyrics = example.get("lyrics", "[inst]")\n        lang_code = example.get("language", detect_language(lyrics))\n        lang_display = LANG.get(lang_code, LANG["en"])\n        print(f"🎲 Random: {caption[:50]}... | Language: {lang_display}")\n        return caption, lyrics, lang_display\n\n    def gen_music_ui(tags, lyrics, dur, lang, ntracks, think, seed, bpm, key, ts, steps, guide, instrumental, p=gr.Progress()):\n        if not tags: raise gr.Error("Style/Tags required!")\n        final_lyrics = "[inst]" if instrumental else (lyrics or "[inst]")\n        s, r = (-1, True)\n        if seed and str(seed).strip() not in ["", "-1"]:\n            try: s, r = int(seed), False\n            except: pass\n        params = GenerationParams(caption=tags, lyrics=final_lyrics, thinking=think, duration=float(dur),\n                    vocal_language=lc(lang), inference_steps=int(steps), guidance_scale=float(guide),\n                    seed=s, bpm=int(bpm) if bpm else None, keyscale=key, timesignature=ts, task_type="text2music")\n        config = GenerationConfig(batch_size=int(ntracks), use_random_seed=r, audio_format="mp3")\n        p(0.2, desc="Generating...")\n        res = generate_music(dit_handler=dit, llm_handler=llm, params=params, config=config, save_dir=OUTPUT_DIR)\n        if not res.success: raise gr.Error(res.error)\n        paths = [a.get("path") for a in res.audios if isinstance(a, dict) and a.get("path")]\n        return (paths[0] if paths else None, paths[1] if len(paths) > 1 else None, f"Generated {len(paths)} track(s)")\n\n    def gen_cover_ui(ref, tags, lyr, strength, dur, lang, n, p=gr.Progress()):\n        if not ref: raise gr.Error("Upload audio!")\n        params = GenerationParams(caption=tags, lyrics=lyr or "[inst]", thinking=True, duration=float(dur),\n                    vocal_language=lc(lang), task_type="cover", reference_audio=ref, audio_cover_strength=float(strength))\n        config = GenerationConfig(batch_size=int(n), audio_format="mp3")\n        p(0.2, desc="Generating cover...")\n        res = generate_music(dit_handler=dit, llm_handler=llm, params=params, config=config, save_dir=OUTPUT_DIR)\n        if not res.success: raise gr.Error(res.error)\n        paths = [a.get("path") for a in res.audios if isinstance(a, dict) and a.get("path")]\n        return (paths[0] if paths else None, paths[1] if len(paths) > 1 else None, f"Generated {len(paths)} cover(s)")\n\n    def gen_repaint_ui(src, tags, lyr, start, end, strength, dur, lang, p=gr.Progress()):\n        if not src: raise gr.Error("Upload audio!")\n        params = GenerationParams(caption=tags, lyrics=lyr or "[inst]", thinking=True, duration=float(dur),\n                    vocal_language=lc(lang), task_type="repaint", src_audio=src,\n                    repainting_start=float(start), repainting_end=float(end), audio_cover_strength=float(strength))\n        config = GenerationConfig(batch_size=1, audio_format="mp3")\n        p(0.2, desc="Repainting...")\n        res = generate_music(dit_handler=dit, llm_handler=llm, params=params, config=config, save_dir=OUTPUT_DIR)\n        if not res.success: raise gr.Error(res.error)\n        paths = [a.get("path") for a in res.audios if isinstance(a, dict) and a.get("path")]\n        return (paths[0] if paths else None, "Repaint complete")\n\n    def format_ui(cap, lyr):\n        res = format_sample(llm_handler=llm, caption=cap, lyrics=lyr)\n        if not res.success: raise gr.Error(res.error)\n        return (res.caption, res.lyrics, f"BPM: {res.bpm} Key: {res.keyscale}")\n\n    CSS = """@import url(\'https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap\');\n* { font-family: \'Inter\', sans-serif !important; }\n.gradio-container { max-width: 1000px !important; margin: auto !important; }\n.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 25px; border-radius: 15px; margin-bottom: 25px; box-shadow: 0 10px 25px rgba(0,0,0,0.15); }\n.brand-title { color: white; font-size: 2.2em; font-weight: 700; margin: 0 0 10px 0; text-shadow: 2px 2px 4px rgba(0,0,0,0.2); }\n.brand-subtitle { color: #f0f0f0; font-size: 1.1em; margin-bottom: 15px; }\n.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }\n.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 700; font-size: 15px; text-decoration: none; display: inline-block; color: white; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }\n.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }\n.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }\n.x-btn { background: linear-gradient(135deg, #0000 0%, #3333 100%); }\nbutton.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }\n.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }"""\n\n    with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="ACE-Step 1.5 Music Studio | AIQUEST") as demo:\n        gr.HTML(\'<div class="brand-header"><div class="brand-title">🎵 ACE-Step 1.5 Music Studio</div><div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> | AI-Powered Music Creation</div><div class="social-buttons"><a href="https://youtube.com/@aiquestacademy" target="_blank" class="social-btn youtube-btn">▶️ Subscribe on YouTube</a><a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a></div></div>\')\n\n        init_status = gr.Textbox(value="⚠️ Engine Not Started. Click Initialize below.", label="System Status", interactive=False)\n        init_btn = gr.Button("🚀 Initialize Engine (Required First Step)", variant="primary", size="lg")\n\n        with gr.Group(visible=True) as main_ui:\n            with gr.Tabs():\n                with gr.Tab("🎵 Create"):\n                    tags = gr.Textbox(label="🎨 Music Style & Tags", placeholder="e.g., upbeat pop, electronic, female vocals...", lines=2)\n                    with gr.Row():\n                        lang = gr.Dropdown(list(LANG.values()), value="🇬🇧 English", label="🌍 Language (50+ supported)")\n                        instrumental = gr.Checkbox(False, label="🎹 Instrumental Only")\n                    lyr = gr.Textbox(label="📝 Lyrics", lines=6, placeholder="[verse]\\nYour lyrics...\\n\\nClick 🎲 for random example", submit_btn="🎲")\n                    with gr.Row():\n                        dur = gr.Slider(minimum=10, maximum=480, step=10, value=60, label="⏱️ Duration (sec)")\n                        ntracks = gr.Slider(minimum=1, maximum=4, step=1, value=2, label="🎚️ Tracks")\n                    seed = gr.Number(value=-1, label="🎲 Seed (-1 = random, each track unique)", precision=0)\n                    with gr.Accordion("⚙️ Advanced Settings", open=False):\n                        with gr.Row():\n                            bpm = gr.Number(0, label="🥁 BPM (0 = auto)", precision=0)\n                            key = gr.Dropdown(["", "C major", "D major", "E major", "F major", "G major", "A major", "B major", "C minor", "D minor", "E minor"], label="🎹 Key", value="")\n                            ts = gr.Dropdown(["", "4/4", "3/4", "6/8"], label="⏰ Time Signature", value="")\n                        with gr.Row():\n                            steps = gr.Slider(minimum=4, maximum=50, step=1, value=30, label="🔄 Steps")\n                            guide = gr.Slider(minimum=1, maximum=15, step=0.5, value=7.0, label="🎯 Guidance")\n                    think = gr.Checkbox(True, label="💭 Thinking Mode")\n                    btn = gr.Button("🎵 Generate Music", variant="primary", size="lg", interactive=False)\n                    with gr.Row():\n                        out1 = gr.Audio(label="🎧 Track 1")\n                        out2 = gr.Audio(label="🎧 Track 2")\n                    out_info = gr.Textbox(label="ℹ️ Generation Info", interactive=False)\n                    lyr.submit(randomize_example, outputs=[tags, lyr, lang])\n                    btn.click(gen_music_ui, [tags, lyr, dur, lang, ntracks, think, seed, bpm, key, ts, steps, guide, instrumental], [out1, out2, out_info])\n\n                with gr.Tab("🎸 Cover / Remix"):\n                    gr.Markdown("### Upload audio to transform into a new style")\n                    ref = gr.Audio(label="🎵 Reference Audio", type="filepath")\n                    with gr.Row():\n                        ctags = gr.Textbox(label="🎨 Target Style", placeholder="rock, energetic, electric guitar...")\n                        cstr = gr.Slider(minimum=0, maximum=1, step=0.05, value=0.5, label="🎚️ Cover Strength")\n                    clyr = gr.Textbox(label="📝 New Lyrics (Optional)", lines=4, placeholder="Leave empty to keep original")\n                    cbtn = gr.Button("🎸 Generate Cover", variant="primary", size="lg", interactive=False)\n                    with gr.Row():\n                        cout1 = gr.Audio(label="🎧 Cover 1")\n                        cout2 = gr.Audio(label="🎧 Cover 2")\n                    cout_info = gr.Textbox(label="ℹ️ Info", interactive=False)\n                    cbtn.click(gen_cover_ui, [ref, ctags, clyr, cstr, dur, lang, ntracks], [cout1, cout2, cout_info])\n\n                with gr.Tab("🎨 Repaint"):\n                    gr.Markdown("### Regenerate a specific section")\n                    rsrc = gr.Audio(label="🎵 Source Audio", type="filepath")\n                    rtags = gr.Textbox(label="🎨 Repaint Style", placeholder="jazz, smooth, saxophone...")\n                    rlyr = gr.Textbox(label="📝 Section Lyrics", lines=3)\n                    with gr.Row():\n                        rstart = gr.Number(0, label="⏮️ Start (seconds)", precision=1)\n                        rend = gr.Number(-1, label="⏭️ End (-1 = end)", precision=1)\n                        rstr = gr.Slider(minimum=0, maximum=1, step=0.05, value=0.7, label="🎚️ Strength")\n                    rbtn = gr.Button("🎨 Repaint Region", variant="primary", size="lg", interactive=False)\n                    rout_audio = gr.Audio(label="🎧 Result")\n                    rout_info = gr.Textbox(label="ℹ️ Info", interactive=False)\n                    rbtn.click(gen_repaint_ui, [rsrc, rtags, rlyr, rstart, rend, rstr, dur, lang], [rout_audio, rout_info])\n\n                with gr.Tab("📝 Format Lyrics"):\n                    gr.Markdown("### Auto-format lyrics and detect metadata")\n                    fcap = gr.Textbox(label="🎨 Style", placeholder="pop, upbeat...")\n                    flyr = gr.Textbox(label="📝 Raw Lyrics", lines=6, placeholder="Your unformatted lyrics...")\n                    fbtn = gr.Button("✨ Format & Enhance", variant="primary", size="lg", interactive=False)\n                    fcap_out = gr.Textbox(label="✨ Enhanced Style", interactive=False)\n                    flyr_out = gr.Textbox(label="📝 Formatted Lyrics", lines=6, interactive=False)\n                    fmeta_out = gr.Textbox(label="📊 Detected Metadata", interactive=False)\n                    fbtn.click(format_ui, [fcap, flyr], [fcap_out, flyr_out, fmeta_out])\n\n        gr.HTML(\'<div class="footer"><p style="font-size: 16px; margin: 5px 0;">🎵 Created by <strong>AIQuest Academy</strong></p><p style="font-size: 14px; margin: 5px 0; color: #9ca3af;">Free & Open Source | MIT License | ACE-Step 1.5 (0.6B Model)</p><p style="font-size: 13px; margin: 10px 0;"><a href="https://youtube.com/@aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">YouTube</a> | <a href="https://x.com/aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">X (Twitter)</a></p></div>\')\n\n        init_btn.click(wrap_init_engine, outputs=[init_status, btn, init_btn]).then(lambda: [gr.update(interactive=True)]*3, outputs=[cbtn, rbtn, fbtn])\n\n    return demo\n\nimport acestep\nfound = False\ntarget_func = "create_gradio_interface"\npath = getattr(acestep, "__path__", [])\nfor _, name, _ in pkgutil.walk_packages(path, prefix="acestep."):\n    try:\n        mod = importlib.import_module(name)\n        if hasattr(mod, target_func):\n            print(f"🎯 Patching {target_func} in {mod.__name__}...")\n            def wrapper(*args, **kwargs):\n                dit = next((a for a in args if "AceStepHandler" in str(type(a))), args[0] if args else None)\n                llm = next((a for a in args if "LLMHandler" in str(type(a))), args[1] if len(args) > 1 else None)\n                return run_custom_ui(dit, llm, **kwargs)\n            setattr(mod, target_func, wrapper)\n            found = True\n            break\n    except ImportError: pass\n\nif not found:\n    print("⚠️ Fallback Patch...")\n    try:\n        import acestep.gradio_ui.interface\n        acestep.gradio_ui.interface.create_gradio_interface = lambda *a, **k: run_custom_ui(a[0], a[1], **k)\n        print("🎯 Fallback successful!")\n    except: print("❌ Failed to patch UI.")\n\ntry:\n    import tomllib\nexcept ImportError:\n    import tomli as tomllib\n\nwith open("pyproject.toml", "rb") as f:\n    pyproject = tomllib.load(f)\n\n# DO NOT pass --lm_model_path here! It causes model download before Gradio launches.\n# The custom UI\'s wrap_init_engine() handles 0.6B download + init when user clicks "Initialize Engine".\nsys.argv = ["acestep", "--share"]\n\nscripts = pyproject.get("project", {}).get("scripts", {})\nentry = scripts.get("acestep", "")\n\nif entry and ":" in entry:\n    mod_path, func = entry.rsplit(":", 1)\n    print(f"🚀 Running CLI: {mod_path}:{func}")\n    m = importlib.import_module(mod_path)\n    getattr(m, func)()\nelse:\n    import runpy\n    runpy.run_module("acestep", run_name="__main__")\n'

with open("/content/ACE-Step-1.5/custom_launch.py", "w") as f:
    f.write(LAUNCHER)
print("📝 Custom launcher written")
print("🎵 Launching AIQuest Academy Music Studio...")
print("🔗 Watch for the public Gradio URL!\n")

!MPLBACKEND=agg uv run python custom_launch.py

/content/ACE-Step-1.5
📝 Custom launcher written
🎵 Launching AIQuest Academy Music Studio...
🔗 Watch for the public Gradio URL!

Loaded configuration from /content/ACE-Step-1.5/.env.example (fallback)
Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
/content/ACE-Step-1.5/.venv/lib/python3.12/site-packages/torchao/quantization/quant_api.py:2525: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
2026-03-05 12:54:00.484 | WARNING  | acestep.training.trainer:<module>:40 - bitsandbytes not installed. Using standard AdamW.
🎯 Patching create_gradio_interface in acestep.acestep_v15_pipeline...
🚀 Running CLI: acestep.acestep_v15_pipeline:main
2026-03-05 12:54:03.984 | INFO     | acestep.gpu_config:get_gpu_memory_gb:453 - CUDA GPU detected: Tesla T4 (14.6 GB)

GPU Con

---

<div align="center">

### 🎉 Enjoyed this notebook?

If this was helpful, please **⭐ star the repo** and **subscribe** for more free AI tools & tutorials!

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/▶%20Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20𝕏-0000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

**Made with ❤️ by AIQUEST** | [@aiquestacademy](https://youtube.com/@aiquestacademy)

</div>